In [1]:
import polars as pl
import bz2
import gzip as gz
import matplotlib.pyplot as plt
import numpy as np
import re
from collections import Counter

In [ ]:
# df = pl.scan_parquet("/home/ma/a/alb25/Project/thesis_code/data/raw/*")

In [ ]:
# Adding the user type columns

def add_user_type(column):
    return (pl.when(pl.col(column).str.contains(r"^U\d+@")).then(pl.lit("human"))
        .when(pl.col(column).str.contains(r"^C\d+\$@")).then(pl.lit("machine"))
        .when(pl.col(column).str.contains(r"^(SYSTEM|LOCAL SERVICE|NETWORK SERVICE)@")).then(pl.lit("system"))
        .when(pl.col(column).str.contains(r"^ANONYMOUS LOGON@")).then(pl.lit("anon"))
        .otherwise(pl.lit("other")))

df = df.with_columns([add_user_type("source_user@domain").alias("source_user_type"),
    add_user_type("destination_user@domain").alias("destination_user_type")])

In [5]:
# converting the tiem column to a good format
df = df.with_columns(pl.col('time').cast(pl.Int64).alias('time'))
df = df.with_columns(pl.duration(seconds=pl.col('time')).alias('time'))

In [ ]:
df = df.with_columns(pl.when(pl.col('success/failure') == 'Success').then(True).otherwise(False).alias('success')).drop('success/failure')

In [ ]:

#TODO convert logon type, authentication type, and authentication orientation to categorical variables

In [ ]:
df.sink_parquet("/home/ma/a/alb25/Project/thesis_code/data/processed/auth_w_user_type.parquet")

time,source_user@domain,destination_user@domain,source_computer,destination_computer,authentication_type,logon_type,authentication_orientation,success/failure,source_user_type,destination_user_type
duration[μs],str,str,str,str,str,str,str,str,str,str
57d 23h 59m 59s,"""U7813@DOM1""","""U7813@DOM1""","""C5618""","""C5618""","""?""","""Network""","""LogOff""","""Success""","""human""","""human"""
57d 23h 59m 59s,"""U8712@DOM1""","""U8712@DOM1""","""C18568""","""C1065""","""?""","""?""","""TGS""","""Success""","""human""","""human"""
57d 23h 59m 59s,"""U939@DOM1""","""U939@DOM1""","""C10""","""C10""","""?""","""Network""","""LogOff""","""Success""","""human""","""human"""
57d 23h 59m 59s,"""U9@?""","""U9@?""","""C222""","""C222""","""?""","""?""","""TGT""","""Fail""","""human""","""human"""
57d 23h 59m 59s,"""U9@DOM1""","""U9@DOM1""","""C222""","""C222""","""Negotiate""","""Interactive""","""LogOn""","""Fail""","""human""","""human"""


In [11]:
df.select('authentication_orientation').unique().collect()

authentication_orientation
str
"""ScreenLock"""
"""ScreenUnlock"""
"""AuthMap"""
"""LogOn"""
"""TGT"""
"""LogOff"""
"""TGS"""


In [21]:
df.select('authentication_type').unique().collect()

authentication_type
str
"""MICROSOFT_AUTHENTICATION_PACKA…"
"""MICROSOFT_AUTHENTICATION_PACKA"""
"""Negotiate"""
"""MICROSOFT_AUTHENTICATION_PA"""
"""MICROSOFT_AUTHENTICATION_PACKA…"
…
"""ACRONIS_RELOGON_AUTHENTICATION…"
"""MICROSOFT_AUTHENTICATION_PACKA…"
"""Wave"""


In [22]:
df.select('logon_type').unique().collect()

logon_type
str
"""NetworkCleartext"""
"""Unlock"""
"""Batch"""
"""Network"""
"""RemoteInteractive"""
"""Service"""
"""?"""
"""Interactive"""
"""CachedInteractive"""
